# Warcoffe 13 — Text-to-Draft: Klasifikasi Kategori Menu

Notebook ini melatih dan membandingkan dua model untuk memprediksi kategori menu dari kalimat transaksi:

1. **TF-IDF (word + char n-gram) + Logistic Regression**
2. **TF-IDF (word + char n-gram) + Linear SVC**

Kelas yang dipakai sama dengan kategori di halaman Kasir:

- `Minuman Kemasan`
- `Minuman Racikan / Sachet`
- `Makanan`

Evaluasi memakai dataset latih sintetis dari 57 menu resmi dan **manual test set** yang ditulis terpisah agar tidak bocor ke data latih.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from IPython.display import Markdown, display

from app.ml.dataset import generate_training_samples, load_manual_test
from app.ml.training import (
    MODEL_LABELS,
    build_pipeline,
    comparison_rows,
    train_and_evaluate,
)

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## 1. Dataset

Dataset latih dibuat dari 57 item menu Warcoffe 13. Setiap item menghasilkan beberapa variasi kalimat:

- nama item saja
- nama + kuantitas
- prefix penjualan (`laku`, `jual`, `terjual`, `pesan`)
- varian suhu (`panas`, `dingin`, `es`)
- harga eksplisit (`Rp10.000`, `10rb`)
- deskripsi item
- typo ringan pada sebagian sampel

Manual test set berisi 40 kalimat yang ditulis manual dan tidak dipakai saat training.

In [2]:
dataset = generate_training_samples(seed=42)
manual_test = load_manual_test()

print(f"Total sampel latih : {len(dataset)}")
print(f"Total manual test  : {len(manual_test)}")
display(dataset["label"].value_counts().rename("jumlah").to_frame())
display(manual_test["label"].value_counts().rename("jumlah").to_frame())

Total sampel latih : 1293
Total manual test  : 40


,jumlah
label,
Minuman Racikan / Sachet,524
Minuman Kemasan,453
Makanan,316


,jumlah
label,
Minuman Racikan / Sachet,15
Makanan,14
Minuman Kemasan,11


In [3]:
dataset.groupby("label").head(4)[["text", "label", "template"]]

,text,label,template
0,1 teh botol original,Minuman Kemasan,qty_name
1,laku teh botol original 2x,Minuman Kemasan,prefix_name_qty
2,minuman kaleng teh botol original,Minuman Kemasan,category_name
3,jual teh botol original 2x.,Minuman Kemasan,prefix_name_qty
453,layu kopi hitam tiga.,Minuman Racikan / Sachet,prefix_name_qty
454,kopi hitam tiga panas,Minuman Racikan / Sachet,name_qty_variant
455,kopi hitam tiga hangat.,Minuman Racikan / Sachet,name_qty_variant
456,kopi hitam seduh hangat,Minuman Racikan / Sachet,description
977,mie goeng + telur 11rb,Makanan,name_price
978,Mie goreng + telur tiga,Makanan,name_variant_qty


## 2. Arsitektur Model

Kedua model memakai representasi fitur yang sama:

- **Word TF-IDF**: `ngram_range=(1, 2)`, `sublinear_tf=True`
- **Char TF-IDF**: `analyzer="char_wb"`, `ngram_range=(3, 5)`, `sublinear_tf=True`
- Gabungan fitur: `FeatureUnion`
- `class_weight="balanced"` untuk menangani jumlah kelas yang tidak seimbang

Perbedaannya hanya pada classifier:

- **Logistic Regression**: menghasilkan `predict_proba` sehingga confidence lebih mudah ditafsirkan.
- **Linear SVC**: margin-based, biasanya lebih cepat, tetapi tidak memiliki probabilitas bawaan.

In [4]:
for key, label in MODEL_LABELS.items():
    pipeline = build_pipeline(key)
    print(f"{key}: {label}")
    print(pipeline)
    print("-" * 80)

logreg: TF-IDF (word + char n-gram) + Logistic Regression
Pipeline(steps=[('features',
                 FeatureUnion(transformer_list=[('word',
                                                 TfidfVectorizer(ngram_range=(1,
                                                                              2),
                                                                 strip_accents='unicode',
                                                                 sublinear_tf=True,
                                                                 token_pattern='(?u)\\b\\w+\\b')),
                                                ('char',
                                                 TfidfVectorizer(analyzer='char_wb',
                                                                 ngram_range=(3,
                                                                              5),
                                                                 strip_accents='unicode',
                         

## 3. Training dan Evaluasi

Evaluasi dilakukan pada tiga skenario:

1. **Random split 80/20** — mengukur performa pada variasi kalimat dengan item yang pernah dilihat.
2. **Group split 80/20 berdasarkan item** — mengukur generalisasi ke nama menu yang belum pernah dilihat.
3. **Manual test set** — mengukur performa pada kalimat yang ditulis terpisah.

Ditambah **5-fold StratifiedGroupKFold cross-validation** dengan grup berupa item menu agar tidak ada kebocoran data.

In [5]:
result = train_and_evaluate(seed=42)
print("Training selesai.")
print("Model terbaik:", result["metadata"]["best_model_label"])

Training selesai.
Model terbaik: TF-IDF (word + char n-gram) + Logistic Regression


## 4. Tabel Perbandingan

In [6]:
comparison = pd.DataFrame(comparison_rows(result))
comparison = comparison.rename(
    columns={
        "model": "Model",
        "random_accuracy": "Random Acc",
        "random_macro_f1": "Random Macro-F1",
        "group_accuracy": "Group Acc",
        "group_macro_f1": "Group Macro-F1",
        "manual_accuracy": "Manual Acc",
        "manual_macro_f1": "Manual Macro-F1",
        "cv_macro_f1": "CV Macro-F1",
        "cv_std": "CV Std",
        "latency_ms": "Latency (ms/sample)",
    }
)
display(comparison.round(4))

,Model,Random Acc,Random Macro-F1,Group Acc,Group Macro-F1,Manual Acc,Manual Macro-F1,CV Macro-F1,CV Std,Latency (ms/sample)
0,TF-IDF (word + char n-gram) + Logistic Regression,0.9846,0.9863,0.6792,0.6713,1.0,1.0,0.6977,0.1359,0.1246
1,TF-IDF (word + char n-gram) + Linear SVC,0.9807,0.9829,0.6528,0.6591,1.0,1.0,0.6943,0.1328,0.1121


### Classification report pada group split

Group split adalah skenario yang paling ketat karena nama menu di test set tidak muncul di training set.

In [7]:
for key, experiment in result["experiments"].items():
    print(experiment.label)
    report = pd.DataFrame(experiment.group_split.report).T
    display(report.round(3))
    print()

TF-IDF (word + char n-gram) + Logistic Regression


,precision,recall,f1-score,support
Makanan,0.956,0.662,0.782,65.000
Minuman Kemasan,0.521,0.440,0.477,84.000
Minuman Racikan / Sachet,0.671,0.862,0.755,116.000
accuracy,0.679,0.679,0.679,0.679
macro avg,0.716,0.655,0.671,265.000
weighted avg,0.693,0.679,0.673,265.000



TF-IDF (word + char n-gram) + Linear SVC


,precision,recall,f1-score,support
Makanan,1.000,0.662,0.796,65.000
Minuman Kemasan,0.459,0.464,0.462,84.000
Minuman Racikan / Sachet,0.664,0.784,0.719,116.000
accuracy,0.653,0.653,0.653,0.653
macro avg,0.708,0.637,0.659,265.000
weighted avg,0.681,0.653,0.657,265.000


### Confusion matrix pada group split

In [8]:
labels = sorted(result["group_test"]["label"].unique().tolist())
for key, experiment in result["experiments"].items():
    matrix = pd.DataFrame(
        experiment.group_split.confusion_matrix,
        index=labels,
        columns=labels,
    )
    print(experiment.label)
    display(matrix)
    print()

TF-IDF (word + char n-gram) + Logistic Regression


,Makanan,Minuman Kemasan,Minuman Racikan / Sachet
Makanan,43,20,2
Minuman Kemasan,0,37,47
Minuman Racikan / Sachet,2,14,100



TF-IDF (word + char n-gram) + Linear SVC


,Makanan,Minuman Kemasan,Minuman Racikan / Sachet
Makanan,43,21,1
Minuman Kemasan,0,39,45
Minuman Racikan / Sachet,0,25,91


### Classification report pada manual test

In [9]:
for key, experiment in result["experiments"].items():
    print(experiment.label)
    report = pd.DataFrame(experiment.manual_test.report).T
    display(report.round(3))
    print()

TF-IDF (word + char n-gram) + Logistic Regression


,precision,recall,f1-score,support
Makanan,1.0,1.0,1.0,14.0
Minuman Kemasan,1.0,1.0,1.0,11.0
Minuman Racikan / Sachet,1.0,1.0,1.0,15.0
accuracy,1.0,1.0,1.0,1.0
macro avg,1.0,1.0,1.0,40.0
weighted avg,1.0,1.0,1.0,40.0



TF-IDF (word + char n-gram) + Linear SVC


,precision,recall,f1-score,support
Makanan,1.0,1.0,1.0,14.0
Minuman Kemasan,1.0,1.0,1.0,11.0
Minuman Racikan / Sachet,1.0,1.0,1.0,15.0
accuracy,1.0,1.0,1.0,1.0
macro avg,1.0,1.0,1.0,40.0
weighted avg,1.0,1.0,1.0,40.0


## 5. Kesimpulan dan Alasan Pemilihan Model

In [10]:
best_key = result["best_key"]
best = result["experiments"][best_key]
other_key = "linearsvc" if best_key == "logreg" else "logreg"
other = result["experiments"][other_key]

display(
    Markdown(
        f'''
### Model terpilih: **{best.label}**

| Metrik | {best.label} | {other.label} |
|---|---:|---:|
| Random split accuracy | {best.random_split.accuracy:.4f} | {other.random_split.accuracy:.4f} |
| Random split macro-F1 | {best.random_split.macro_f1:.4f} | {other.random_split.macro_f1:.4f} |
| Group split accuracy | {best.group_split.accuracy:.4f} | {other.group_split.accuracy:.4f} |
| Group split macro-F1 | {best.group_split.macro_f1:.4f} | {other.group_split.macro_f1:.4f} |
| Manual test accuracy | {best.manual_test.accuracy:.4f} | {other.manual_test.accuracy:.4f} |
| Manual test macro-F1 | {best.manual_test.macro_f1:.4f} | {other.manual_test.macro_f1:.4f} |
| CV macro-F1 | {best.cv_macro_f1_mean:.4f} ± {best.cv_macro_f1_std:.4f} | {other.cv_macro_f1_mean:.4f} ± {other.cv_macro_f1_std:.4f} |
| Latency | {best.manual_test.latency_ms:.2f} ms/sample | {other.manual_test.latency_ms:.2f} ms/sample |

**Alasan pemilihan:**

1. Pada manual test, kedua model mencapai macro-F1 {best.manual_test.macro_f1:.4f}. Jika seri, pemilihan dilanjutkan ke group split.
2. Pada group split, model terpilih memiliki macro-F1 {best.group_split.macro_f1:.4f}, sedikit lebih tinggi daripada {other.group_split.macro_f1:.4f}.
3. Logistic Regression menyediakan `predict_proba`, sehingga confidence dapat ditampilkan langsung ke pengguna.
4. Selisih latency tidak signifikan untuk skala warung, sehingga akurasi dan interpretabilitas confidence lebih diprioritaskan.
5. Model tetap ringan karena hanya TF-IDF + linear classifier, tanpa GPU dan tanpa API eksternal.

Model terbaik disimpan ke `models/best_model.joblib` dan dipakai oleh service FastAPI.
'''
    )
)


### Model terpilih: **TF-IDF (word + char n-gram) + Logistic Regression**

| Metrik | TF-IDF (word + char n-gram) + Logistic Regression | TF-IDF (word + char n-gram) + Linear SVC |
|---|---:|---:|
| Random split accuracy | 0.9846 | 0.9807 |
| Random split macro-F1 | 0.9863 | 0.9829 |
| Group split accuracy | 0.6792 | 0.6528 |
| Group split macro-F1 | 0.6713 | 0.6591 |
| Manual test accuracy | 1.0000 | 1.0000 |
| Manual test macro-F1 | 1.0000 | 1.0000 |
| CV macro-F1 | 0.6977 ± 0.1359 | 0.6943 ± 0.1328 |
| Latency | 0.12 ms/sample | 0.11 ms/sample |

**Alasan pemilihan:**

1. Pada manual test, kedua model mencapai macro-F1 1.0000. Jika seri, pemilihan dilanjutkan ke group split.
2. Pada group split, model terpilih memiliki macro-F1 0.6713, sedikit lebih tinggi daripada 0.6591.
3. Logistic Regression menyediakan `predict_proba`, sehingga confidence dapat ditampilkan langsung ke pengguna.
4. Selisih latency tidak signifikan untuk skala warung, sehingga akurasi dan interpretabilitas confidence lebih diprioritaskan.
5. Model tetap ringan karena hanya TF-IDF + linear classifier, tanpa GPU dan tanpa API eksternal.

Model terbaik disimpan ke `models/best_model.joblib` dan dipakai oleh service FastAPI.


## 6. Uji Inference Model Tersimpan

Model yang disimpan dimuat oleh service dan dipakai untuk memprediksi kategori dari kalimat baru.

In [11]:
from app.ml.predictor import get_predictor

predictor = get_predictor()
assert predictor is not None, "Model belum tersedia. Jalankan sel training di atas."

examples = [
    "laku kopi liong panas 2",
    "teh botol 1",
    "indomie telur 2",
    "pocari sweat",
    "kentang goreng",
    "milo dingin",
]

for text in examples:
    prediction = predictor.predict(text)
    print(f"{text:35s} -> {prediction.label:28s} ({prediction.confidence:.3f})")

laku kopi liong panas 2             -> Minuman Racikan / Sachet     (0.969)
teh botol 1                         -> Minuman Kemasan              (0.943)
indomie telur 2                     -> Makanan                      (0.985)
pocari sweat                        -> Minuman Kemasan              (0.953)
kentang goreng                      -> Makanan                      (0.970)
milo dingin                         -> Minuman Racikan / Sachet     (0.899)


## 7. Integrasi ke Website

Service FastAPI memakai model ini pada endpoint:

```text
POST /api/v1/parse-transaction
```

Alur integrasi:

```text
Next.js /ai-assistant
  -> /api/ai/parse-transaction (proxy)
  -> FastAPI /api/v1/parse-transaction
  -> model TF-IDF + linear classifier
  -> draft JSON
  -> konfirmasi pemilik
```

Jalankan service:

```bash
cd ai-service
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
uvicorn app.main:app --reload --port 8000
```

Lalu jalankan Next.js:

```bash
npm run dev
```